In [1]:
pip install jieba rank_bm25 sentence-transformers faiss-cpu

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
     ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
     ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
     ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
     - -------------------------------------- 0.5/19.2 MB 1.4 MB/s eta 0:00:14
     --- ------------------------------------ 1.6/19.2 MB 3.3 MB/s eta 0:00:06
     ------- -------------------------------- 3.4/19.2 MB 5.1 MB/s eta 0:00:04
     ----------- ---------------------------- 5.5/19.2 MB 6.5 MB/s eta 0:00:03
     --------------- ------------------------ 7.6/19.2 MB 7.2 MB/s eta 0:00:02
     -------------------- ------------------- 10.0/19.2 MB 7.9 MB/s eta 0:00:02
     -------------------------- ------------- 12.6/19.2 MB 8.5 MB/s eta 0:00:01
     ------------------------------- -------- 14.9/19.2 MB 8.9 MB/s eta 0:00:01
     -----------

  DEPRECATION: Building 'jieba' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'jieba'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [2]:
import pandas as pd
import numpy as np
import jieba
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss
import pickle
import os
import time

# ==========================================
# 1. 路径配置 (严格遵循项目规范)
# ==========================================
PROCESSED_DATA_PATH = "./data/processed/tablet_corpus.csv"
INDEX_DIR = "./indices"

# 确保索引保存目录存在
os.makedirs(INDEX_DIR, exist_ok=True)

BM25_INDEX_PATH = os.path.join(INDEX_DIR, "bm25_index.pkl")
FAISS_INDEX_PATH = os.path.join(INDEX_DIR, "faiss_index.bin")

# ==========================================
# 2. 加载专属知识库 (Corpus)
# ==========================================
print(f"正在加载知识库: {PROCESSED_DATA_PATH} ...")
df_corpus = pd.read_csv(PROCESSED_DATA_PATH)
corpus_texts = df_corpus['review'].tolist()
print(f"✅ 成功加载 {len(corpus_texts)} 条文档用于构建索引。")

# ==========================================
# 3. 构建稀疏检索索引 (BM25)
# 对应论文：2.2.1 稀疏检索：BM25 算法原理
# ==========================================
print("\n开始构建 BM25 稀疏索引...")
start_time = time.time()

# 使用结巴分词对中文进行切词
tokenized_corpus = [list(jieba.cut(doc)) for doc in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)

# 将构建好的 BM25 模型保存到本地
with open(BM25_INDEX_PATH, "wb") as f:
    pickle.dump(bm25, f)
    
print(f"✅ BM25 索引构建完成并保存！耗时: {time.time() - start_time:.2f} 秒")

# ==========================================
# 4. 构建稠密检索索引 (FAISS + Bi-Encoder)
# 对应论文：2.2.2 稠密检索：Bi-Encoder 架构
# ==========================================
print("\n开始加载 BGE 向量模型 (初次运行会自动下载，请耐心等待)...")
# 我们使用 BAAI 开源的顶尖中文 embedding 模型
model_name = "BAAI/bge-small-zh-v1.5"
encoder = SentenceTransformer(model_name)

print("正在对知识库进行稠密向量化 (Encode)...")
start_time = time.time()
# 将所有文本转换为向量，normalize_embeddings=True 推荐用于内积计算相似度
corpus_embeddings = encoder.encode(corpus_texts, show_progress_bar=True, normalize_embeddings=True)
print(f"✅ 向量化完成！耗时: {time.time() - start_time:.2f} 秒")

print("正在构建 FAISS 向量数据库...")
# 获取向量的维度 (BGE-small 为 512 维)
embedding_dim = corpus_embeddings.shape[1]
# 使用内积 (Inner Product) 初始化 FAISS 索引，因为向量已经归一化，内积等价于余弦相似度
faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(corpus_embeddings)

# 将 FAISS 索引保存到本地
faiss.write_index(faiss_index, FAISS_INDEX_PATH)
print(f"✅ FAISS 索引构建完成并保存至 {FAISS_INDEX_PATH}！")

# ==========================================
# 5. 快速测试 (Sanity Check)
# ==========================================
print("\n" + "="*40)
print("🔍 索引快速测试：双路召回对比")
print("="*40)

test_query = "平板看视频屏幕清晰吗？电池耐用吗？"
print(f"测试 Query: '{test_query}'\n")

# -- 测试 BM25 召回 --
tokenized_query = list(jieba.cut(test_query))
bm25_top_n = bm25.get_top_n(tokenized_query, corpus_texts, n=3)
print("【BM25 (关键词匹配) Top-3 召回结果】:")
for i, doc in enumerate(bm25_top_n):
    print(f"  {i+1}. {doc}")

# -- 测试 FAISS 召回 --
query_embedding = encoder.encode([test_query], normalize_embeddings=True)
# search 返回距离 (相似度) 和对应的索引 ID
distances, indices = faiss_index.search(query_embedding, 3)

print("\n【FAISS (语义匹配) Top-3 召回结果】:")
for i, idx in enumerate(indices[0]):
    print(f"  {i+1}. [相似度: {distances[0][i]:.4f}] {corpus_texts[idx]}")

Building prefix dict from the default dictionary ...


正在加载知识库: ./data/processed/tablet_corpus.csv ...
✅ 成功加载 9920 条文档用于构建索引。

开始构建 BM25 稀疏索引...


Dumping model to file cache C:\Users\24201\AppData\Local\Temp\jieba.cache
Loading model cost 0.663 seconds.
Prefix dict has been built successfully.


✅ BM25 索引构建完成并保存！耗时: 2.13 秒

开始加载 BGE 向量模型 (初次运行会自动下载，请耐心等待)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

正在对知识库进行稠密向量化 (Encode)...


Batches:   0%|          | 0/310 [00:00<?, ?it/s]

✅ 向量化完成！耗时: 45.67 秒
正在构建 FAISS 向量数据库...
✅ FAISS 索引构建完成并保存至 ./indices\faiss_index.bin！

🔍 索引快速测试：双路召回对比
测试 Query: '平板看视频屏幕清晰吗？电池耐用吗？'

【BM25 (关键词匹配) Top-3 召回结果】:
  1. 平板电脑拿在手里看视频玩游戏都很好用，屏幕清晰，电池耐用，挺好的。
  2. 不是有送保护套外壳吗？怎么就裸机了？这不是欺骗消费者吗？
  3. 我想问下我8月8日买的平板，里面怎么又7月15日的视频？请解释一下，有这么坑人的吗？都说支持国产，支持国货，这个能让人相信吗？

【FAISS (语义匹配) Top-3 召回结果】:
  1. [相似度: 0.8097] 平板好用，看视频也很清晰，电池的电充满了能待机很长时间。
  2. [相似度: 0.8008] 平板电脑拿在手里看视频玩游戏都很好用，屏幕清晰，电池耐用，挺好的。
  3. [相似度: 0.7922] 平板很好，屏幕很大，也清晰。看电视眼睛一点都不累！
